# Kinematics: Theta2 and Theta3

For any given Altitude and Roll pointing orientation, the inverse kinematics calculates the required Theta2 and Theta3 motor angles to achive the desired result. 

This notebook analyses the relationships between Altitude, Roll and Theta2, Theta3.


In [1]:
import pandas as pd

from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import os, sys
import importlib
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import kinematics 
importlib.reload(kinematics)
from kinematics import azaltroll_to_theta, theta_to_azaltroll, altitude_to_maxroll


Simulate a grid of pointing orientations spanning IK Altitude and Roll as well as FK Theta2 and Theta3 positions at a given Az

In [2]:

az = 180
d = []
for y in range(-80,81,5):
    for x in range(-80,81,5):
        f_t2 = x
        t1, t2, t3 = azaltroll_to_theta(az, x, y)
        _, alt, roll = theta_to_azaltroll(az, x, y)
        if x<-8:
            alt, roll, f_t2 = 0, 0, 0
        d.append({ 
            "i_alt": x, "i_roll": y, "i_theta2": t2, "i_theta3": t3,
            "f_alt": alt, "f_roll": roll, "f_theta2": f_t2, "f_theta3": y,
        })
d=pd.DataFrame(d)
d.columns


Index(['i_alt', 'i_roll', 'i_theta2', 'i_theta3', 'f_alt', 'f_roll',
       'f_theta2', 'f_theta3'],
      dtype='object')

# Altitude vs Theta2 and Theta3


In [3]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Altitude vs Theta2", "Altitude vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_alt", color="i_roll")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_alt", color="i_roll")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Roll (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Altitude (deg)", row=1, col=1)
fig.update_yaxes(title_text="Altitude (deg)", row=1, col=2)

fig.show()

# Roll vs Theta2 and Theta3

In [4]:
fig = make_subplots(
    rows=1, 
    cols=2,
    subplot_titles=("Roll vs Theta2", "Roll vs Theta3")
)

# First plot (theta2)
fig1 = px.line(d, x="i_theta2", y="i_roll", color="i_alt")
for trace in fig1.data:
    trace.legendgroup = trace.name
    trace.showlegend = True   # Only show legend once
    fig.add_trace(trace, row=1, col=1)

# Second plot (theta3)
fig2 = px.line(d, x="i_theta3", y="i_roll", color="i_alt")
for trace in fig2.data:
    trace.legendgroup = trace.name
    trace.showlegend = False  # Hide duplicate legend entries
    fig.add_trace(trace, row=1, col=2)

# Adjust overall layout
fig.update_layout(height=800, width=1600, showlegend=True, legend_title_text="Altitude (deg)")

fig.update_xaxes(title_text="Theta 3 (deg)", row=1, col=2)
fig.update_xaxes(title_text="Theta 2 (deg)", row=1, col=1)

fig.update_yaxes(title_text="Roll (deg)", row=1, col=1)
fig.update_yaxes(title_text="Roll (deg)", row=1, col=2)

fig.show()

# Altitude and Roll isobars in the Theta2/Theta3 space

In [5]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["i_alt"].unique():
    df_sub = d[d["i_alt"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"alt={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["i_roll"].unique():
    df_sub = d[d["i_roll"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["i_theta3"], y=df_sub["i_theta2"], name=f"roll={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1600, xaxis_title="Theta3 (deg)", yaxis_title="Theta2 (deg)", legend_title_text="Grouping"
)

fig.show()

# Theta2 and Theta3 isobars in the Altitude/Roll  space

In [6]:
fig = go.Figure()

# --- SOLID LINES: grouped by alt ---
for val in d["f_theta2"].unique():
    df_sub = d[d["f_theta2"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta2={val}",
            mode="lines", line=dict(dash="solid"), legendgroup="alt", showlegend=True)
    )

# --- DASHED LINES: grouped by roll ---
for val in d["f_theta3"].unique():
    df_sub = d[d["f_theta3"] == val]
    fig.add_trace(
        go.Scatter(x=df_sub["f_roll"], y=df_sub["f_alt"], name=f"theta3={val}",
            mode="lines", line=dict(dash="dash"), legendgroup="roll", showlegend=True)
    )

fig.update_layout(
    height=900, width=1400, xaxis_title="Roll (deg)", yaxis_title="Altitude (deg)", legend_title_text="Grouping"
)

fig.show()

# Reachable Altitude and Roll

In [28]:
import numpy as np
import plotly.graph_objects as go

# ── Valid physical theta2 range, from kinematics.py's q_to_theta() ───────────
THETA2_MIN, THETA2_MAX = -8.0, 81.5

def _solve_theta3(alt, roll, theta2_signed):
    """theta3 (deg) needed to hit (alt, roll) for a given signed theta2 candidate.
    Returns None if geometrically impossible for this theta2."""
    t2r = np.radians(theta2_signed)
    sin_t2 = np.sin(t2r)
    if abs(sin_t2) < 1e-9:
        return 0.0 if abs(alt) < 1e-6 else None
    ratio = np.sin(np.radians(alt)) / sin_t2
    if abs(ratio) > 1 + 1e-9:
        return None
    ratio = np.clip(ratio, -1, 1)
    t3_mag = np.degrees(np.arccos(ratio))
    if roll == 0:
        return t3_mag
    sign_theta2 = np.sign(theta2_signed) if theta2_signed != 0 else 1.0
    needed_sin_sign = -np.sign(roll) * sign_theta2
    return t3_mag if needed_sin_sign >= 0 else -t3_mag

def classify_azaltroll(alt, roll):
    """
    Classify a target (alt, roll) as:
      'unflipped'    - reachable with |theta3| <= 90 deg (no rear pan-flip needed)
      'flipped_only' - only reachable via the theta3 ~180 deg flip
      'unreachable'  - not reachable at all, even with a flip
    theta2 = arccos(cos(alt)*cos(roll)) has two candidate signed solutions
    (+theta2, -theta2); both are checked against the real valid range
    THETA2_MIN..THETA2_MAX, and against the theta3 required to actually hit
    the target roll sign.
    """
    theta2_pos = np.degrees(np.arccos(np.clip(np.cos(np.radians(alt)) * np.cos(np.radians(roll)), -1, 1)))
    candidates = []
    for t2 in (theta2_pos, -theta2_pos):
        if THETA2_MIN <= t2 <= THETA2_MAX:
            t3 = _solve_theta3(alt, roll, t2)
            if t3 is not None:
                candidates.append((t2, t3))
    if not candidates:
        return "unreachable"
    t2_best, t3_best = min(candidates, key=lambda c: abs(c[1]))
    return "unflipped" if abs(t3_best) <= 90.0 else "flipped_only"

# ── Build the Alt x Roll grid and classify every point ───────────────────────
N_ALT, N_ROLL = 381, 361
alt_vals = np.linspace(-20, 85, N_ALT)
roll_vals = np.linspace(-85, 85, N_ROLL)
ALT, ROLL = np.meshgrid(alt_vals, roll_vals, indexing="ij")

UNFLIPPED = np.zeros_like(ALT, dtype=bool)
FLIPPED_ONLY = np.zeros_like(ALT, dtype=bool)
THETA2_REQ = np.degrees(np.arccos(np.clip(np.cos(np.radians(ALT)) * np.cos(np.radians(ROLL)), -1, 1)))

for i in range(ALT.shape[0]):
    for j in range(ALT.shape[1]):
        kind = classify_azaltroll(ALT[i, j], ROLL[i, j])
        if kind == "unflipped":
            UNFLIPPED[i, j] = True
        elif kind == "flipped_only":
            FLIPPED_ONLY[i, j] = True

# ── Colors (dark theme) ───────────────────────────────────────────────────────
BG          = "#0e1117"
PBG         = "#331f1f"
GRID_CLR    = "rgba(255,255,255,0.08)"
TEXT_CLR    = "#e6e6e6"
GREEN_DIM   = "rgba(51,209,122,0.35)"
GREEN_LABEL = "#33d17a"
FLIP_DIM    = "rgba(180,180,190,0.18)"
FLOOR_LINE  = "#ff8a3d"
ISO_LINE    = "rgba(255,255,255,0.35)"

fig = go.Figure()

# 1) Region only reachable via the theta3 flip (context, dim)
Z_FLIP = np.where(FLIPPED_ONLY, 1.0, np.nan)
fig.add_trace(go.Heatmap(
    x=roll_vals, y=alt_vals, z=Z_FLIP,
    zmin=0, zmax=1, colorscale=[[0, FLIP_DIM], [1, FLIP_DIM]],
    showscale=False, hoverinfo="skip", zsmooth=False, showlegend=False,
))

# 2) Unflipped reachable region (the main answer)
Z_UNFLIP = np.where(UNFLIPPED, 1.0, np.nan)
fig.add_trace(go.Heatmap(
    x=roll_vals, y=alt_vals, z=Z_UNFLIP,
    zmin=0, zmax=1, colorscale=[[0, GREEN_DIM], [1, GREEN_DIM]],
    showscale=False, hoverinfo="skip", zsmooth=False, showlegend=False,
))

# 3) theta2-required iso-contours, labeled with their degree value
fig.add_trace(go.Contour(
    x=roll_vals, y=alt_vals, z=THETA2_REQ,
    contours=dict(start=10, end=THETA2_MAX-1, size=10, coloring="none",
                  showlabels=True, labelfont=dict(size=18, color=ISO_LINE)),
    line=dict(color=ISO_LINE, width=1, dash="dot"),
    showscale=False, hoverinfo="skip", showlegend=False,
))

# 4) Hard theta2 = THETA2_MAX boundary curve
fig.add_trace(go.Contour(
    x=roll_vals, y=alt_vals, z=THETA2_REQ,
    contours=dict(start=THETA2_MAX, end=THETA2_MAX, size=1, coloring="none",
                  showlabels=True, labelfont=dict(size=22, color=FLOOR_LINE), labelformat=".1f"),
    line=dict(color=FLOOR_LINE, width=3),
    showscale=False, hoverinfo="skip", showlegend=False,
))

fig.add_hline(y=0, line=dict(color="rgba(255,255,255,0.25)", width=1))
fig.add_vline(x=0, line=dict(color="rgba(255,255,255,0.25)", width=1))

fig.update_layout(
    title=dict(
        text="Benro Polaris — Reachable Alt / Roll Envelope",
        x=0.02, xanchor="left",
        font=dict(color=TEXT_CLR, size=42, family="Arial, sans-serif"),
    ),
    paper_bgcolor=BG, plot_bgcolor=PBG,
    font=dict(color=TEXT_CLR, family="Arial, sans-serif", size=22),
    width=1920, height=1080,
    margin=dict(l=90, r=60, t=110, b=80),
    showlegend=False,
    xaxis=dict(title="Roll (degrees)", range=[-90, 90], dtick=20,
               gridcolor=GRID_CLR, zeroline=False, showline=True, linecolor="rgba(255,255,255,0.3)",
               ticks="outside", tickcolor="rgba(255,255,255,0.3)"),
    yaxis=dict(title="Altitude (degrees)", range=[-20, 90], dtick=10,
               gridcolor=GRID_CLR, zeroline=False, showline=True, linecolor="rgba(255,255,255,0.3)",
               ticks="outside", tickcolor="rgba(255,255,255,0.3)"),
    annotations=[
        dict(x=0, y=45, xref="x", yref="y", text="Reachable Envelope", showarrow=False,
             font=dict(color=GREEN_LABEL, size=42)),
        dict(x=-45, y=-10, xref="x", yref="y", text="Reachable only via M3 flip", showarrow=False,
             font=dict(color=FLOOR_LINE, size=22)),
        dict(x=81.5, y=15, xref="x", yref="y",
             text=f"M2 = {THETA2_MAX:.1f}\u00b0 boundary", showarrow=True, ax=-120, ay=-60,
             font=dict(color=FLOOR_LINE, size=22), arrowcolor=FLOOR_LINE),
    ],
)

fig.show()

In [7]:
max_theta2 = 81.5
alts = list(range(-8, 90, 2))
max_rolls = [altitude_to_maxroll(a, max_theta2) for a in alts]


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=max_rolls2,
    y=alts,
    mode='lines+markers',
    name=f'Max Roll Angle°)',
    line=dict(color='green', width=2),
    marker=dict(size=5),
    hovertemplate='Alttitude: %{y}°<br>Max Roll Angle: %{x:.1f}°<extra></extra>'
))


fig.update_layout(
    title=f'Maximum achievable Roll vs Altitude (limited by Theta2 = {max_theta2}°)',
    xaxis_title='Max Roll Angle (deg)',
    yaxis_title='Altitude (deg)',
    xaxis=dict(range=[0, 85], dtick=10),
    yaxis=dict(range=[-10, 92], dtick=10, zeroline=True),
    width=800,
    height=500,
    template='plotly_dark',
    hovermode='x unified'
)

NameError: name 'max_rolls2' is not defined